# NB_HCHistorical_Bronze_To_Silver
**Source:** `AI_Medallion_Bronze.dbo.HC_Bronze_Historical`  
**Target:** `AI_Medallion.Silver.hc_silver_historical`  
**Load type:** Full (overwrite)

In [ ]:
import os
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
# After kernel restart: run this cell, bootstrap, then read/transform/write cells.
# read_table() caches parquet under ~/.fabric/medallion_cache/ (Windows: %USERPROFILE%\.fabric\medallion_cache\).
# Re-read from OneLake (ignore cache): os.environ["MEDALLION_CACHE_REFRESH"] = "1"
# Delete ALL local cache: remove the medallion_cache folder (PowerShell: Remove-Item -Recurse -Force $env:USERPROFILE\.fabric\medallion_cache)
# Delete ONE table cache: remove its subfolder (e.g. AI_Medallion__Silver__cursor_usage).
# Disable caching: os.environ["MEDALLION_LOCAL_TABLE_CACHE"] = "0"


In [ ]:
%run ./common/bootstrap


In [18]:
apply_local_merge_spark_conf(spark)
bronze = read_table_broken_lineage(TABLE_HC_BRONZE, "hc_bronze")
country_lookup = break_lineage_local(
    load_country_lookup(spark, TABLE_COUNTRY), "country_lookup"
)

StatementMeta(, ae404b8e-60bb-4c4b-a500-fc245a4c345c, 13, Finished, Available, Finished, False)

In [19]:
hc = bronze.select(
    F.col("`Month`").cast("int").alias("Month_No"),
    F.col("`Person ID`").alias("OktaUserEmployeeNumber"),
    F.col("`Employee Code`").alias("HCEmployeeNumber"),
    F.col("`Fullname [Employee]`").alias("OktaUserFullName"),
    F.col("`Region [Employee]`").alias("HCRegion"),
    F.col("`Country [Employee]`").alias("_raw_country"),
    F.col("`Office [Employee]`").alias("HCOffice"),
    F.col("`Business Unit [Employee]`").alias("HCBusinessUnit"),
    F.col("`Division [Employee]`").alias("HCDivision"),
    F.col("`Department [Employee]`").alias("HCDepartment"),
    F.col("`Legal Entity [Employee]`").alias("HCLegalEntity"),
    F.col("`Grid Subgroup [Employee]`").alias("HCGridSubgroup"),
    F.col("`Talent Segment [Employee]`").alias("HCTalentSegment"),
    F.col("`Job Title [Employee]`").alias("HCJobTitle"),
    F.lower(F.trim(F.col("`Email [Employee]`"))).alias("OktaUserEmail"),
    F.col("`User Network [Employee]`").alias("HCUserNetwork"),
    F.col("`Inc. Roles`").alias("HCIncRoles"),
    F.col("`Okta Region`").alias("OktaUserRegion"),
    F.col("`Division INC`").alias("HCDivisionINC"),
    F.col("`Total Employees`").cast("int").alias("HCTotalEmployees"),
    F.col("`Area Emeal`").alias("HCAreaEmeal"),
    F.col("KeyIdEmployeeCountry").alias("HCKeyIdEmployeeCountry"),
    F.col("`Business Unit Code [Employee]`").alias("HCBusinessUnitCode"),
    F.col("`Replace Mapping`").alias("HCReplaceMapping"),
)

hc = with_normalized_country(hc, "_raw_country", country_lookup).drop("_raw_country")

hc = hc.withColumn("HCDate", date_from_yyyymm(F.col("Month_No")))
hc = hc.withColumn(
    "HCKeyDivision",
    F.concat(F.col("HCDivisionINC"), F.lit("/"), F.col("OktaUserCountry")),
)
hc = hc.withColumn(
    "HCKeyEmail",
    F.concat(F.col("Month_No").cast("string"), F.lit("/"), F.col("OktaUserEmail")),
)

# Business keys: Month_No + OktaUserEmail should be unique per spec; enforce not null on required fields
hc = hc.filter(
    F.col("Month_No").isNotNull()
    & F.col("OktaUserEmail").isNotNull()
    & F.col("HCDate").isNotNull()
    & F.col("HCKeyDivision").isNotNull()
    & F.col("HCKeyEmail").isNotNull()
)

StatementMeta(, ae404b8e-60bb-4c4b-a500-fc245a4c345c, 14, Finished, Available, Finished, False)

In [20]:
write_full_table(hc, TABLE_HC_SILVER)

show_sample(hc.limit(10))

StatementMeta(, ae404b8e-60bb-4c4b-a500-fc245a4c345c, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d59932cc-73d4-4c89-91f5-1678b6aa791a)